# Optimización del calendario de la Premier League

Este notebook formula y resuelve un calendario doble Round-Robin con Gurobi, usando la instancia de equipos seleccionada en [modelo_calendario.py](../modelo_calendario.py). El CSV histórico identifica los equipos; las listas de clásicos, estadios compartidos, clubes europeos y jornadas críticas se configuran según los datos disponibles.

## Formulación entera

Sean $E$ el conjunto de equipos y $J=\{1,\ldots,2(|E|-1)\}$ el conjunto de jornadas. $P$ contiene los pares de clásicos, $S$ los pares de equipos que comparten estadio, $E^{UEFA}$ los equipos con compromisos europeos y $J^c$ las jornadas críticas. Se definen variables binarias $x_{ijt}$ (el equipo $i$ recibe a $j$ en $t$) y $y_{it}$ (el equipo $i$ juega de local en $t$). Las variables $b^H_{it}$ y $b^A_{it}$ indican breaks consecutivos de local y visita entre $t-1$ y $t$; $v_{it}$ indica un viaje en jornada crítica. Estas últimas son continuas acotadas entre 0 y 1, pero las restricciones las fuerzan a sus valores enteros.

$$\min\quad w_b\sum_{i\in E}\sum_{t=2}^{|J|}(b^H_{it}+b^A_{it})+w_v\sum_{i\in E^{UEFA}}\sum_{t\in J^c}v_{it}$$

Sujeto a:

- Un partido por equipo y jornada: $\sum_{j\ne i}(x_{ijt}+x_{jit})=1$ para todo $i,t$.
- Cada localía de cada pareja ordenada exactamente una vez: $\sum_{t\in J}x_{ijt}=1$ para todo $i\ne j$.
- Definición de localía y balance: $y_{it}=\sum_{j\ne i}x_{ijt}$ y $\sum_{t\in J}y_{it}=|J|/2$.
- Break local: $b^H_{it}\ge y_{i,t-1}+y_{it}-1$, $b^H_{it}\le y_{i,t-1}$ y $b^H_{it}\le y_{it}$.
- Break visitante: $b^A_{it}\ge 1-y_{i,t-1}-y_{it}$, $b^A_{it}\le 1-y_{i,t-1}$ y $b^A_{it}\le 1-y_{it}$.
- No más de dos localías seguidas: $y_{i,t-1}+y_{it}+y_{i,t+1}\le 2$.
- A lo más $K$ clásicos por jornada: $x_{ijt}+x_{jit}\le K$ para cada par en $P$ y jornada $t$.
- Equipos que comparten estadio no juegan de local simultáneamente: $y_{it}+y_{jt}\le 1$ para cada par en $S$.
- Viaje en jornada crítica: $v_{it}=1-y_{it}$ para $i\in E^{UEFA}, t\in J^c$.
- $x,y$ binarias; $0\le b^H,b^A,v\le1$.

Se resuelve primero con parámetros predeterminados de Gurobi y luego con cortes y heurísticas desactivados. Ambas ejecuciones tienen un límite de 3600 segundos y reportan incumbente, cota, gap, tiempo y nodos.

In [ ]:
# Localizar e importar el modelo general compartido.
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

_directorios = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
RUTA_PROYECTO = next(
    (directorio for directorio in _directorios if (directorio / "modelo_calendario.py").is_file()),
    None,
)
if RUTA_PROYECTO is None:
    raise FileNotFoundError("No se encontró modelo_calendario.py en el proyecto ni sus carpetas padre.")
sys.path.insert(0, str(RUTA_PROYECTO))

from modelo_calendario import resolver_instancia

URLS_TEMPORADA = {
    "2021-2022": "https://www.football-data.co.uk/mmz4281/2122/E0.csv",
    "2022-2023": "https://www.football-data.co.uk/mmz4281/2223/E0.csv",
    "2023-2024": "https://www.football-data.co.uk/mmz4281/2324/E0.csv",
    "2024-2025": "https://www.football-data.co.uk/mmz4281/2425/E0.csv",
    "2025-2026": "https://www.football-data.co.uk/mmz4281/2526/E0.csv",
    "2026-2027": "https://www.football-data.co.uk/mmz4281/2627/E0.csv",
}
TEMPORADA = "2024-2025"
URL_CSV = URLS_TEMPORADA[TEMPORADA]
print(f"Instancia seleccionada: {TEMPORADA}")

Instancia seleccionada: 2024-2025


In [ ]:
# Configurar restricciones operativas y pesos de la función objetivo.
PARAMETROS_MODELO = {
    "clasicos": [],  # Ejemplo: [("Arsenal", "Tottenham")]
    "estadios_compartidos": [],
    "equipos_europeos": [],
    "jornadas_criticas": [],  # Números entre 1 y 38 para una liga de 20 equipos
    "max_clasicos_por_jornada": 1,
    "peso_breaks": 1.0,
    "peso_viajes": 1.0,
    "tiempo_limite": 3600,
    "numero_equipos_esperado": 20,
}

In [ ]:
# Resolver con parámetros por defecto y con cortes/heurísticas desactivados.
configuraciones = {
    "Normal (Gurobi por defecto)": False,
    "Sin cortes ni heurísticas": True,
}
resultados = {}
for nombre, desactivar_cortes_heuristicas in configuraciones.items():
    resultados[nombre] = resolver_instancia(
        TEMPORADA,
        URL_CSV,
        desactivar_cortes_heuristicas=desactivar_cortes_heuristicas,
        **PARAMETROS_MODELO,
    )

# Reporte solicitado para ambas ejecuciones.
reporte = pd.DataFrame(
    [
        {
            "Configuración": nombre,
            "Estado Gurobi": resultado["estado"],
            "Objetivo (ObjVal)": resultado["objetivo"],
            "Cota (ObjBoundC)": resultado["cota"],
            "Gap (MIPGap)": resultado["gap"],
            "Tiempo (s)": resultado["tiempo"],
            "Nodos (NodeCount)": resultado["nodos"],
        }
        for nombre, resultado in resultados.items()
    ]
)
display(reporte)

# Mostrar también el calendario de la primera configuración si se encontró solución.
resultado_normal = resultados["Normal (Gurobi por defecto)"]
print(f"Partidos leídos desde internet: {len(resultado_normal['partidos_historicos'])}")
print(f"Breaks: {resultado_normal['breaks']}")
print(f"Viajes críticos: {resultado_normal['viajes_criticos']}")
display(resultado_normal["calendario"])
display(resultado_normal["partidos_historicos"][["Date", "HomeTeam", "AwayTeam"]].head(10))